In [12]:
import cv2
import streamlit as st
from streamlit_webrtc import webrtc_streamer, WebRtcMode
import av
import numpy as np

In [13]:
# Load face cascade classifier
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')


In [14]:
#Home Page
def home():
    # Home page content
    st.title("Real-Time Face Detection")
    st.write("""
        This app detects faces in real-time using OpenCV and Streamlit.
        It uses the Haar cascade classifier to predict faces.
    """)

In [15]:
def video_frame_callback(frame: av.VideoFrame) -> av.VideoFrame:
    img = frame.to_ndarray(format="bgr24")
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(img_gray, scaleFactor=1.3, minNeighbors=5)
    for (x, y, w, h) in faces:
        cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)

    return av.VideoFrame.from_ndarray(img, format="bgr24")


In [16]:
#Webcam face detection
def video_face_detection():
    st.title("Webcam Face Detection")
    webrtc_streamer(
    key="face-detection",
    mode=WebRtcMode.SENDRECV,
    rtc_configuration={"iceServers": [{"urls": ["stun:stun.l.google.com:19302"]}]},
    video_frame_callback=video_frame_callback,
    media_stream_constraints={"video": True, "audio": False},
    async_processing=True)


In [17]:

#Image Face Detection
def image_face_detection():
    # Image face detection page content
    st.title("Image Face Detection")
    st.write("Upload an image file to detect faces.")
    uploaded_image = st.file_uploader('Upload Image', type=["jpg", "jpeg", "png"])
    if uploaded_image is not None:
        image = cv2.imdecode(np.frombuffer(uploaded_image.read(), np.uint8), 1)
        img_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(img_gray, scaleFactor=1.1, minNeighbors=5)
        for (x, y, w, h) in faces:
            cv2.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 2)
        st.image(image, channels="BGR")

In [18]:
def main():
    # Streamlit app layout
    st.sidebar.title("Navigation")
    page = st.sidebar.selectbox("Select Page", ["Home", "Image Face Detection", "Webcam Face Detection"])

    if page == "Home":
        home()
    elif page == "Image Face Detection":
        image_face_detection()
    elif page == "Webcam Face Detection":
        video_face_detection()

if __name__ == "__main__":
    main()